# 01 - Place the gated datasets (ThyroidXL + Stanford AIMI)

**These two datasets cannot be downloaded automatically.** They are gated for IRB and
data-use-agreement reasons, and this repository redistributes no pixel of either. You
request access yourself, accept the terms, and then **drop your approved copy into the
exact folder shown below**. This notebook downloads nothing: it tells you where the files
go and then validates that you put them there correctly.

| Dataset | Where to request access |
|---|---|
| **ThyroidXL** (MICCAI 2025) | Gated Hugging Face dataset: <https://huggingface.co/datasets/hunglc007/ThyroidXL> - request access, accept the terms, then download the repository (`repo_type="dataset"`). |
| **Stanford AIMI Thyroid Cine-clip** (Radiology AI 2022) | Stanford AIMI portal: <https://aimi.stanford.edu/datasets/thyroid-ultrasound-cine-clip> - register, sign the DUA, and download. You receive a single HDF5 bundle. |

Access approval is not instant for either one, so request both before you plan to run
anything. The two open datasets (DDTI, TN3K) need none of this; notebook `00` fetches them
for you.


In [ ]:
# Always run from the repository root so every relative path resolves.
import os
from pathlib import Path
while not (Path.cwd() / 'pyproject.toml').exists() and Path.cwd() != Path.cwd().parent:
    os.chdir('..')
assert (Path.cwd() / 'pyproject.toml').exists(), 'run this notebook from inside the repo'
print('repo root:', Path.cwd())

## Where to drop your copies

Two moves, one per dataset. The preprocessing scripts hardcode these names, so the
spelling and capitalisation matter.

**ThyroidXL** - the Hugging Face dataset already has the `train/` and `test/` layout.
Put the extracted folder at `data/raw/extracted/ThyroidXL/`.

**Stanford AIMI** - rename the HDF5 you downloaded to `dataset.hdf5` and put it at
`data/raw/extracted/Stanford/dataset.hdf5`.

The complete tree, with every level written out:

```
data/raw/extracted/
├── ThyroidXL/
│   ├── train/
│   │   ├── images/                 *.png   (one image per file)
│   │   ├── masks/                  *.png   (same filename as its image in images/)
│   │   └── train_annotations.json
│   └── test/
│       ├── images/                 *.png
│       ├── masks/                  *.png
│       └── test_annotations.json
└── Stanford/
    └── dataset.hdf5
```

Two details the scripts depend on:

- A mask must carry **exactly the same filename** as the image it belongs to. Pairing is
  by filename, so a mask whose name does not match is silently skipped.
- The annotation JSONs are keyed by patient, in the shape
  `{"info": {"<patient_id>": {"images": ["<filename>.png", ...]}, ...}}`. That mapping is
  what makes the splits patient-level, so the splits cannot be built without it.

The Stanford HDF5 holds four datasets of equal length, one row per frame:
`image` (uint8 frames), `mask` (uint8, thresholded at 127), `annot_id` (byte string) and
`frame_num` (byte string). If your download uses different key names, it is a different
packaging of the dataset and preprocessing will not read it.

`annot_id` identifies a **nodule**, not a patient: the release contains 192 annotated
nodules from 167 patients and exposes no patient identifier. The splits are therefore
grouped by nodule, which is the finest grouping the distributed file supports; every frame
of a given nodule stays in one split, so no clip is ever cut across train and test.

Everything under `data/raw/` is git-ignored, so your gated copies can never be committed.


## Validate placement

Run the cell below before moving on. It checks the folders the preprocessing scripts
actually read, not just the metadata files, and reports counts against the published
totals: **ThyroidXL 11,635 images / 4,093 patients**, **Stanford AIMI 17,412 frames /
192 nodules**.


In [ ]:
import json
import h5py

rows = []
RAW = Path('data/raw/extracted')

# --- ThyroidXL -------------------------------------------------------------
# Check both the annotation JSONs (which make_splits.py reads) AND the image/mask
# folders (which preprocess_thyroidxl.py reads). Valid JSON alongside empty or
# misnamed folders is the failure this catches.
tx = RAW / 'ThyroidXL'
ann = {s: tx / s / f'{s}_annotations.json' for s in ('train', 'test')}
dirs = {(s, k): tx / s / k for s in ('train', 'test') for k in ('images', 'masks')}
missing = [str(p) for p in list(ann.values()) + list(dirs.values()) if not p.exists()]
if missing:
    rows.append(('ThyroidXL', 'MISSING', 'not found: ' + ', '.join(missing[:3])
                 + (f' (+{len(missing)-3} more)' if len(missing) > 3 else '')))
else:
    a = {s: json.loads(ann[s].read_text()) for s in ('train', 'test')}
    n_pat = sum(len(a[s]['info']) for s in a)
    listed = {fn for s in a for rec in a[s]['info'].values() for fn in rec.get('images', [])}
    n_img = len(listed)
    on_disk = {p.name for s in ('train', 'test') for p in dirs[(s, 'images')].glob('*.png')}
    n_masks = sum(len(list(dirs[(s, 'masks')].glob('*.png'))) for s in ('train', 'test'))
    absent = listed - on_disk
    counts_ok = (n_img == 11635 and n_pat == 4093)
    files_ok = (not absent) and n_masks >= len(on_disk)
    detail = (f'{n_pat} patients / {n_img} images listed (expected 4093 / 11635); '
              f'{len(on_disk)} images and {n_masks} masks on disk')
    if absent:
        detail += f'; {len(absent)} listed files missing from images/, e.g. {sorted(absent)[0]}'
    rows.append(('ThyroidXL', 'OK' if (counts_ok and files_ok) else 'CHECK', detail))

# --- Stanford AIMI ---------------------------------------------------------
# Open the HDF5 and count real frames and real patients, so a truncated or
# differently-packaged file is caught rather than passed.
st = RAW / 'Stanford' / 'dataset.hdf5'
if not st.exists():
    rows.append(('Stanford AIMI', 'MISSING', f'expected the HDF5 at {st}'))
else:
    try:
        with h5py.File(st, 'r') as f:
            required = {'image', 'mask', 'annot_id', 'frame_num'}
            present = set(f.keys())
            if not required.issubset(present):
                rows.append(('Stanford AIMI', 'CHECK',
                             f'missing HDF5 keys {sorted(required - present)}; '
                             f'found {sorted(present)}'))
            else:
                n_frames = f['image'].shape[0]
                lengths = {k: f[k].shape[0] for k in required}
                n_nod = len({a.decode().strip() for a in f['annot_id'][:]})
                same_len = len(set(lengths.values())) == 1
                ok = (n_frames == 17412 and n_nod == 192 and same_len)
                detail = f'{n_nod} nodules / {n_frames} frames (expected 192 / 17412)'
                if not same_len:
                    detail += f'; unequal dataset lengths {lengths}'
                rows.append(('Stanford AIMI', 'OK' if ok else 'CHECK', detail))
    except OSError as e:
        rows.append(('Stanford AIMI', 'CHECK', f'could not open the HDF5: {e}'))

MARK = {'OK': 'OK  ', 'CHECK': 'CHECK', 'MISSING': 'MISS'}
print(f"{'dataset':<15}{'status':<8}detail")
print('-' * 92)
for name, status, detail in rows:
    print(f'{name:<15}{MARK[status]:<8}{detail}')


If both rows read **OK**, continue to `02_preprocess.ipynb`.

**MISS** means the files are not where the scripts look - re-read the tree above; the most
common cause is one extra nesting level left over from unzipping.

**CHECK** means the data is there but does not match the published totals. Preprocessing
will still run, but your numbers will not be comparable to the reported ones, so re-download
before proceeding rather than after.
